# 개인 맞춤 학습 코치 에이전트를 ReAct + Tool + Gemini 조합으로 작성.

구성은 2파일:
- curriculum.json : 커리큘럼/로드맵(미니 버전)
- app.py : LangChain Agent + Tools + Gemini 코드

In [ ]:
!pip install langchain langchain-google-genai google-genai langchain-chroma langchain-community
!pip install langchain-classic
!pip install sentence-transformers python-dotenv

In [ ]:
import os
import json
from typing import Dict, Any, Optional

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain_classic.agents import create_tool_calling_agent
from langchain_classic.agents import AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

load_dotenv()

True

In [ ]:
# 1) 커리큘럼 로드 (JSON 파일에서 한 번만 읽어오기)
CURRICULUM_PATH = "curriculum.json"

def load_curriculum() -> Dict[str, Any]:
    if not os.path.exists(CURRICULUM_PATH):
        raise FileNotFoundError(f"{CURRICULUM_PATH} 파일을 찾을 수 없습니다.")
    with open(CURRICULUM_PATH, "r", encoding="utf-8") as f:
        return json.load(f)

CURRICULUM = load_curriculum()

# track 키 목록 (python / data / ml)
VALID_TRACKS = list(CURRICULUM["tracks"].keys())

def _track_exists(track: str) -> bool:
    return track in CURRICULUM["tracks"]

In [ ]:
# 2) LLM 준비 (Gemini)
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.3,
)

In [ ]:
# 3) Tool 정의
#    - LangChain @tool 데코레이터를 사용해서 Agent에서 호출 가능하게 함
@tool
def show_track_overview(track: str) -> str:
    """
    주어진 트랙(track)에 대한 전체 개요(설명 + 레벨/토픽)를 반환한다.
    track 값은 'python', 'data', 'ml' 중 하나여야 한다.
    """
    track = track.strip()
    if not _track_exists(track):
        return f"'{track}' 트랙은 존재하지 않습니다. 가능한 값: {', '.join(VALID_TRACKS)}"

    t = CURRICULUM["tracks"][track]
    lines = []
    lines.append(f"[트랙] {t['name']}")
    lines.append(f"[설명] {t['description']}")
    lines.append("")

    for level in t["levels"]:
        lines.append(f"- 레벨 {level['level']}: {level['title']}")
        lines.append("  · 주요 토픽: " + ", ".join(level["topics"]))
        lines.append("")

    return "\n".join(lines)

@tool
def recommend_next_level(track: str, current_level: str) -> str:
    """
    주어진 트랙에서 현재 레벨 이후에 어떤 레벨을 공부하면 좋을지 추천한다.
    current_level 은 숫자 형태의 문자열이어야 한다. 예: '1', '2', '3'
    """
    track = track.strip()
    if not _track_exists(track):
        return f"'{track}' 트랙은 존재하지 않습니다. 가능한 값: {', '.join(VALID_TRACKS)}"

    try:
        cur = int(current_level)
    except ValueError:
        return f"current_level 인자는 숫자 형태의 문자열이어야 합니다. 예: '1', '2', '3'"

    t = CURRICULUM["tracks"][track]
    levels = t["levels"]

    # 현재 레벨보다 큰 레벨 중 가장 낮은 것 찾기
    candidates = [lv for lv in levels if lv["level"] > cur]
    if not candidates:
        return f"{t['name']} 트랙에서 레벨 {cur} 이후의 레벨은 정의되어 있지 않습니다. 이미 마지막 단계일 수 있습니다."

    next_lv = sorted(candidates, key=lambda x: x["level"])[0]
    lines = []
    lines.append(f"[다음 추천 레벨]")
    lines.append(f"- 레벨 {next_lv['level']}: {next_lv['title']}")
    lines.append("· 주요 토픽: " + ", ".join(next_lv["topics"]))
    lines.append("· 추천 실습 과제 예시:")
    for task in next_lv["practice_tasks"]:
        lines.append(f"  - {task}")

    return "\n".join(lines)

@tool
def suggest_practice_by_topic(track: str, topic_keyword: str) -> str:
    """
    특정 트랙에서 topic_keyword(예: '함수', 'Pandas', '전처리')와
    관련이 있어 보이는 레벨/토픽/실습 과제를 모아서 요약해 준다.
    """
    track = track.strip()
    keyword = topic_keyword.strip()

    if not _track_exists(track):
        return f"'{track}' 트랙은 존재하지 않습니다. 가능한 값: {', '.join(VALID_TRACKS)}"

    t = CURRICULUM["tracks"][track]
    lines = [f"트랙: {t['name']}", f"검색 키워드: {keyword}", ""]

    found_any = False

    for lv in t["levels"]:
        # 토픽 중 하나라도 키워드를 포함하면 관련 있다고 간주
        matched_topics = [tp for tp in lv["topics"] if keyword in tp]
        if matched_topics:
            found_any = True
            lines.append(f"- 레벨 {lv['level']}: {lv['title']}")
            lines.append("  · 관련 토픽: " + ", ".join(matched_topics))
            lines.append("  · 실습 과제 예시:")
            for task in lv["practice_tasks"]:
                lines.append(f"    - {task}")
            lines.append("")

    if not found_any:
        return f"키워드 '{keyword}' 와(과) 직접적으로 연결된 토픽을 찾지 못했습니다. 트랙 전체를 먼저 훑어보는 것이 좋겠습니다."

    return "\n".join(lines)

tools = [show_track_overview, recommend_next_level, suggest_practice_by_topic]

In [ ]:
# 4) Agent용 프롬프트
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            (
                "너는 프로그래밍/데이터/머신러닝을 공부하는 학습자를 도와주는 "
                "'개인 맞춤 학습 코치' 에이전트다.\n"
                "사용자의 질문을 읽고, 필요하다면 아래의 Tool을 적절히 사용해라.\n\n"
                "사용 가능한 트랙은 다음과 같다:\n"
                "- python: 파이썬 기초 ~ 함수 ~ 파일 입출력 ~ 클래스\n"
                "- data: Pandas를 중심으로 한 데이터 분석\n"
                "- ml: 머신러닝 입문(지도학습 위주)\n\n"
                "Tool 사용 가이드:\n"
                "- 특정 트랙 전체 구성이나 단계(레벨)를 보여줄 때는 show_track_overview를 사용해라.\n"
                "- '지금 어디까지 배웠는데, 다음에 뭘 해야 하나요?' 같은 질문에는 "
                "  사용자의 설명에서 트랙과 현재 레벨을 추론한 뒤 recommend_next_level을 사용해라.\n"
                "- '함수 연습 문제 추천', 'Pandas 실습 뭐 할까'처럼 특정 키워드에 대한 실습을 찾을 때는 "
                "  suggest_practice_by_topic를 사용해라.\n"
                "- 사용자가 트랙 이름을 직접 말하지 않더라도, 대화 맥락에서 python/data/ml 중 가장 자연스러운 것을 선택해도 된다.\n\n"
                "최종 답변을 만들 때는:\n"
                "1) 먼저 사용자의 현재 상황을 한두 문장으로 정리하고,\n"
                "2) 커리큘럼 정보를 바탕으로 '다음에 공부할 것'을 2~3단계 정도 제안하고,\n"
                "3) 관련 실습 과제나 연습 문제를 bullet 형식으로 3~5개 정도 정리해 주고,\n"
                "4) 마지막에 간단한 응원/학습 조언을 한 문장 덧붙여라.\n"
                "모든 답변은 한국어로 작성하라.\n"
            ),
        ),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

In [ ]:
# 5) Agent + Executor 생성
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=False,        # 내부 동작을 보고 싶으면 True
    max_iterations=3,    # ReAct 루프 최대 3턴 (Tool 여러 번 써도 괜찮게)
)

In [ ]:
# 6) 간단한 대화 루프 (콘솔용)
chat_history = []  # 아주 단순한 형태로만 관리 (실전에서는 ChatMessage 객체 권장)

def ask_coach(question: str):
    print("\n==============================")
    print("질문:", question)

    result = agent_executor.invoke(
        {
            "input": question,
            "chat_history": chat_history,
        }
    )

    answer = result["output"]
    print("\n[학습 코치 답변] : ", answer)

    # 히스토리 업데이트 (여기서는 단순 튜플로만 관리)
    chat_history.append(("human", question))
    chat_history.append(("ai", answer))

In [ ]:
if __name__ == "__main__":
    # 간단 테스트용 질문들
    q1 = "파이썬 기초 문법이랑 if, for 정도는 공부했어. 그 다음에는 뭘 하는 게 좋을까?"
    q2 = "데이터 분석에서 Pandas 맛만 봤는데, 연습 문제나 실습 아이디어를 좀 추천해줘."
    q3 = "머신러닝을 처음 시작하려고 하는데, 어디서부터 시작해서 어떤 순서로 가면 좋을지 알려줘."

    ask_coach(q1)
    ask_coach(q2)
    ask_coach(q3)

    # 간단 인터랙티브 모드
    print("\n--- 직접 질문해보기 (종료: 빈 줄 입력) ---")
    while True:
        q = input("\n질문 입력 > ").strip()
        if not q:
            break
        ask_coach(q)


질문: 파이썬 기초 문법이랑 if, for 정도는 공부했어. 그 다음에는 뭘 하는 게 좋을까?

[학습 코치 답변] :  [{'type': 'text', 'text': '사용자님께서는 파이썬 기초 문법과', 'extras': {'signature': 'CiQBEU0yD3Oe0YDqZmcPS+/gyQc5tELVp0FR+lAQh9UZmQSJNGYKawERTTIP+w0wzpZbZahI64ET+cbLcCByoUGDJSd0GoTct83VaNp7jlWLRnFL6deTZpRIMthDa07cv6TKX0ufTXONPi08ZAbq1KTlLzO1gkylbK0/ioWOf/htiads8yWXLMm6IeSflt8063S3CokCARFNMg+irpFmBsK+kBN5JOz+62xcPFYrxs/CJYsrzx9Ewf4GZvfbthPcMCLMhifnO17fUdpi3Yxwgx8QaKUM8KfSB6i9LrCE5e4T4anzOOBbeByTUv7nH5Moq8NrqD3VxqKV5ASU7q5niQVSdOBnp4qd6pwI/61axmmwOAJTa7Xoe240FIgeruMLFq6zdJZcuBqz5BSWaTWmLdBIvZuBMKn27L4qTQLKYW5BKgA/ot4TSwMXwKGNvREjA5mUgn++ZPVLm61hjIzToH49g2dKLrjHusxhIjVjvPuHAPzhMqUmAcdKdd6WKPnV3oKfF60LKkvEHXD7trXzqwZfxetvYb4fYcQsTWSs4Qr5AQERTTIPFtNCDFEhbOeI6W2In6TXE+bQ3ATlKU3LzeZUBS3DuSIag3/v4SJjd41J4kbTbaDSYWoISV5c/Od8x7RIJRB66usR3gtn2+VN1SRVy99S1trDUUYDTu/3EfNtWq6RbEParSNn65h044Y78laLcqOhINgy6/G7CPaypFrYT3WcazfdSgJImUunktAKdzwuA7kh5us8oWg2+n8RbXn4fLUGURVgmqSOfVLu0s5QTVSMA/nmczGmHEj4wP1ZMfjbwLIbhKyGzUSe+ViVYhEyoG+tOVud5c2gTo3R9dCt4s